In [1]:
import pandas as pd 
import requests

In [2]:
from prophet import Prophet
import sys

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
drug_list = pd.read_csv("prophet_drug_list.csv")
drug_dict = pd.read_csv("drug_dict_with_smiles.csv")

# Normalize for comparison
drug_list["smiles_norm"] = drug_list["smiles"].str.strip()
drug_dict["smiles_norm"] = drug_dict["canonical_smiles"].str.strip()
drug_list["drug_norm"] = drug_list["drug"].str.strip().str.lower()
drug_dict["drug_norm"] = drug_dict["drug"].str.strip().str.lower()

# Match by SMILES or name
smiles_in_prophet = set(drug_list["smiles_norm"].dropna())
names_in_prophet = set(drug_list["drug_norm"].dropna())

drug_dict["in_prophet_smiles"] = drug_dict["smiles_norm"].isin(smiles_in_prophet)
drug_dict["in_prophet_name"] = drug_dict["drug_norm"].isin(names_in_prophet)
drug_dict["in_prophet"] = drug_dict["in_prophet_smiles"] | drug_dict["in_prophet_name"]

print(f"Total drugs in drug_dict:        {len(drug_dict)}")
print(f"Found by SMILES:                 {drug_dict['in_prophet_smiles'].sum()}")
print(f"Found by name only:              {(drug_dict['in_prophet_name'] & ~drug_dict['in_prophet_smiles']).sum()}")
print(f"Found in prophet (total):        {drug_dict['in_prophet'].sum()}")
print(f"Not found in prophet embeddings: {(~drug_dict['in_prophet']).sum()}")

# Drugs with no SMILES at all
no_smiles = drug_dict[drug_dict["canonical_smiles"].isna() | (drug_dict["canonical_smiles"] == "")]
print(f"\nDrugs with no SMILES: {len(no_smiles)}")
print(no_smiles[["dataset", "drug", "chembl_id"]].to_string())

# Fetch missing SMILES from ChEMBL
def fetch_smiles_from_chembl(chembl_id):
    url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/{chembl_id}.json"
    r = requests.get(url)
    if r.status_code == 200:
        return r.json().get("molecule_structures", {}).get("canonical_smiles")
    return None

for idx, row in no_smiles.iterrows():
    if pd.notna(row["chembl_id"]) and row["chembl_id"] != "":
        smiles = fetch_smiles_from_chembl(row["chembl_id"])
        drug_dict.at[idx, "canonical_smiles"] = smiles
        print(f"{row['drug']} -> {smiles}")

Total drugs in drug_dict:        701
Found by SMILES:                 15
Found by name only:              386
Found in prophet (total):        401
Not found in prophet embeddings: 300

Drugs with no SMILES: 52
                  dataset                              drug chembl_id
0                 aissa21                           control       NaN
5                 chang22                           GNE-069       NaN
6                 chang22                           GNE-104       NaN
7                 chang22                           control       NaN
10          mcfarland2020                           BRD3379       NaN
17          mcfarland2020                           control       NaN
27   srivatsan20_sciplex2                           vehicle       NaN
80   srivatsan20_sciplex3                               GSK       NaN
84   srivatsan20_sciplex3              Glesatinib?(MGCD265)       NaN
133  srivatsan20_sciplex3                              Tie2       NaN
135  srivatsan20_sci

In [8]:
ls

INSTALLATION.md  configs/           prophet_mcp_server.py  test/
LICENSE          embeddings/        requirements_mcp.txt   train.sbatch
MANIFEST.in      prophet/           scripts/               tutorials/
README.md        prophet.egg-info/  setup.py


In [13]:
drug_list = pd.read_csv("prophet_drug_list.csv")
drug_dict = pd.read_csv("drug_dict_with_smiles.csv")

drug_list["drug_norm"] = drug_list["drug"].str.strip().str.lower()
drug_list["smiles_norm"] = drug_list["smiles"].str.strip()

drug_dict["drug_norm"] = drug_dict["drug"].str.strip().str.lower()
drug_dict["smiles_norm"] = drug_dict["canonical_smiles"].str.strip()

# Deduplicate both
drug_list_unique = drug_list.drop_duplicates(subset="drug_norm")
drug_dict_dedup = drug_dict.drop_duplicates(subset="drug_norm")[["drug_norm", "chembl_ids"]]

drug_dict_names = set(drug_dict["drug_norm"])
drug_dict_smiles = set(drug_dict["smiles_norm"].dropna())

drug_list_unique["match_name"] = drug_list_unique["drug_norm"].isin(drug_dict_names)
drug_list_unique["match_smiles"] = drug_list_unique["smiles_norm"].isin(drug_dict_smiles)

matched = drug_list_unique[drug_list_unique["match_name"] | drug_list_unique["match_smiles"]].copy()

# Add chembl_id by name
name_to_chembl = drug_dict_dedup.set_index("drug_norm")["chembl_ids"]
matched["chembl_ids"] = matched["drug_norm"].map(name_to_chembl)

print(f"Unique drugs in drug_list: {len(drug_list_unique)}")
print(f"Matched by name:           {drug_list_unique['match_name'].sum()}")
print(f"Matched by SMILES:         {drug_list_unique['match_smiles'].sum()}")
print(f"Total matched:             {len(matched)}")

#matched.to_csv("matched_drugs.csv", index=False)

Unique drugs in drug_list: 136334
Matched by name:           309
Matched by SMILES:         12
Total matched:             316


/tmp/ipykernel_1515256/754507056.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drug_list_unique["match_name"] = drug_list_unique["drug_norm"].isin(drug_dict_names)
/tmp/ipykernel_1515256/754507056.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drug_list_unique["match_smiles"] = drug_list_unique["smiles_norm"].isin(drug_dict_smiles)


In [15]:
list(matched.chembl_ids)

['["CHEMBL3353410"]',
 '["CHEMBL2103875"]',
 nan,
 nan,
 nan,
 '["CHEMBL2028663"]',
 '["CHEMBL325041"]',
 nan,
 nan,
 '["CHEMBL443684"]',
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 '["CHEMBL1173655"]',
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 '["CHEMBL553"]',
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 '["CHEMBL601719"]',
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 '["CHEMBL888"]',
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 '["CHEMBL191334", "CHEMBL407632"]',
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 n

In [8]:
matched = matched.merge(
    drug_dict[["drug_norm", "chembl_id"]].drop_duplicates("drug_norm"),
    on="drug_norm",
    how="left"
)

KeyError: "['drug_norm'] not in index"

In [40]:
drug_dict[drug_dict["drug"] == "Trametinib"]

,dataset,combo_id,drug,gene,source_col,chembl_id,chembl_ids,chembl_ambiguous,drug_org,canonical_smiles,smiles_ref_id,smiles_source
667,tahoe,Trametinib,Trametinib,NaN,drug,NaN,NaN,NaN,Trametinib,CC1=C2C(=C(N(C1=O)C)NC3=C(C=C(C=C3)I)F)C(=O)N(...,Trametinib,pubchem:name


In [42]:
drug_list.columns

Index(['drug', 'smiles', 'drug_norm', 'smiles_norm'], dtype='object')